UPscaledEV Project Main Code
Under the funding from TotalEnergies
Author: Yizhan Gu
Email: yig031@ucsd.edu
Affiliation: UCSD CER

Readme:
This code aims at formulating a complex optimization framework with DERs of building, EV, PV and BESS, to solve a value-stacking problem with wholesale market and demand response market participation. It's based on Yi-An Chen's sclaedev codd and more consice, robust and well-constructed.

Labels:
NOTE: means there's a note and please read it
FIXME: means it's a bug or a problem that needs solving
TODO: means it's a to-do task, but not critical to the code running
VERSION: means there're more than one version for the diversity purpose, possibly shows in objective function choices or results analyses

Acknowledgement:
I gratefully acknowledge the support from my PI Jan Kleissl and TotalEnergies team, and the contributions from Yi-An Chen, whose prior work laid the foundation for this project. Special thanks to the UCSD Grid Lab team members.

Set working path and import packages

In [3]:
import os
os.chdir('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024') 
print("Path is:", os.getcwd(), "\n")
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from datetime import timedelta
from datetime import datetime
import time
from time import process_time
import calendar
import holidays
from pathlib import Path
from random import choices
import cvxpy as cp
import scs
import sys

from forecast_ED_PD import KnownUser, UnKnownUser
import glob



Path is: /Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024 



Paramaters

In [4]:

# Plotting colors
Blue1 = (0.341, 0.82, 1)
Blue2 = (0.341, 0.592, 1)
Blue3 = (0.365, 0.365, 1)
Blue4 = (0, 0, 0.89)
Blue5 = (0, 0, 0.639)
Blue6 = (0, 0.161, 0.42)
Red1 = (1, 0.725, 0.635)
Red2 = (1, 0.6, 0.467)
Red3 = (1, 0.361, 0.353)
Red4 = (1, 0.153, 0.137)
Red5 = (0.812, 0.016, 0)
Red6 = (0.541, 0.008, 0)

# MPC parameters
dt_m_EV = 15
dt_h = dt_m_EV/60
interval = timedelta(minutes=dt_m_EV)
unit = dt_h

# Energy Prices-AL-TOU:
# Summer: off-peak period energy charge rate $0.10679/kW h
#         On-peak period energy charge rate $0.12628/kW h
# Winter: off-peak period energy charge rate $0.09506/kW h
#         On-peak period energy charge rate $0.10626/kW h
c_e_TOU_AL = np.ones((96, 1))*0.10679
c_e_TOU_AL[16*4:21*4] = 0.12628
# TODO: Add Winter rates and auto for days


# Demand Charges
c_NCD = 24.48  #24.48*0.25=6.12
c_PD = 19.14 + 9.78

# Tax
c_tax_DWR = 0.00580                               # x Total kWh
c_tax_oEESbo_Franchise = 0.0688 * c_tax_DWR       # x Total kWh
c_tax_CA_Surcharge = 0.00030                      # x Total kWh
c_tax_CA_Regulatory = 0.00058                     # x Total kWh
# x Total Bill (UDC+Commodity)
c_tax_SD_Franchise = 0.0578
c_tax_all = c_tax_DWR + c_tax_oEESbo_Franchise + c_tax_CA_Surcharge + c_tax_CA_Regulatory

Forecast methods

In [8]:

Numb_EVs = 0
Numb_AbsDiffEVs = 0

Fc_SessionkWh = 'PerfectSessionkWh'  # 'PerfectSessionkWh' #'PersistenceSessionkWh'
Fc_NumbEV = 'PerfectNumbEV'          # 'PerfectNumbEV'     #'PersistenceNumbEV'
Fc_AtArrival = 'PerfectatArrival'    # 'PerfectatArrival'  #'MLatArrival' from Avik
Fc_building = 'Perfect'              # 'Perfectat'  #'Persistence'
Fc_PV = 'Perfect'                    # 'Perfectat'  #'Persistence'

# NOTE: This file is used to get the Dispatch file before Day0 implementation
DAM = 0 # 1/0: w/o day-ahead market participation (Demand Response)

#always run both the base case (no service level reduction) and the case1 with service level reduction
Cases = ['Base', 'Case1']

year = 2024
# year = 2025

Data processing

In [6]:

if Fc_AtArrival == 'MLatArrival':
    User_Data = pd.read_csv("Driver_Table.csv")
    User_known = User_Data['driver_id'][User_Data['TotSession']>10].unique()
    UserNoBess = []


# create weekdays (excluding holidays) and weekends (including holidays) lists
Holidays = holidays.US()
Holidays_dates = list(Holidays.keys())
Holidays_dates = [date for date in Holidays_dates if date.year == year]

datetime(year,1,1) in Holidays
start_ind_Y = datetime(year,1,1)
end_ind_Y = datetime(year+1,1,1)
DateSeries_ThisY = []
while start_ind_Y < end_ind_Y:
    DateSeries_ThisY.append(start_ind_Y)
    start_ind_Y += timedelta(hours=24)

DateSeries_ThisY = pd.Series(DateSeries_ThisY)
Y_Weekends = DateSeries_ThisY[DateSeries_ThisY.dt.dayofweek>=5]
Y_Holidays = DateSeries_ThisY[DateSeries_ThisY.dt.date.isin(Holidays_dates)]

Y_WeekendsWH = pd.concat([Y_Weekends,Y_Holidays],axis=0).drop_duplicates(keep='first', inplace=False).sort_values(axis=0, ascending=True).reset_index(drop=True)
Y_WeekdaysWOH = DateSeries_ThisY[~(DateSeries_ThisY.isin(Y_WeekendsWH))].sort_values(axis=0, ascending=True).reset_index(drop=True)
if len(Y_WeekdaysWOH) + len(Y_WeekendsWH) != len(DateSeries_ThisY):
    print("Error in holiday identification!\n")
    sys.exit()


In [ ]:
# VERSION: 2025 data
if year == 2025:
    # https://docs.google.com/presentation/d/1JRLf_Qb5xuxuLRH_28HUPq5HQBOIKCjCzQMGXw96MlE/edit?slide=id.g308df064212_0_68#slide=id.g308df064212_0_68
    Data = pd.read_csv('2025Data/EV_data/    ', low_memory=False)
    Data['Interval start'] = pd.to_datetime(Data['Interval start'])
    Data['Interval end'] = pd.to_datetime(Data['Interval end'])
    Data = Data[Data['Interval start'].dt.year == 2025]
    
    # filtering
    Data = Data.dropna(subset=['Type', 'Interval kWh', 'Interval average demand kW'])

    print("Number of Intervals in 2025:", len(Data))
    print("Test year 2025:", year)
    print("Date range 2025:", Data['Interval start'].min(), "to", Data['Interval start'].max())
    print("Unique sites 2025:", len(Data['Site Location'].unique()))
    Data['Car_'] = Data['10-digit UID'].astype(str)
# VERSION: 2024 data
elif year == 2024:
    Data = pd.read_csv('UCSD_AllSites_Merge_PostProcessedSession_20210504_20240930.csv', low_memory=False)
    Data['Interval start'] = pd.to_datetime(Data['Interval start'])
    Data['Interval end'] = pd.to_datetime(Data['Interval end'])
    Data['Session start'] = pd.to_datetime(Data['Session start'])
    Data['Session end'] = pd.to_datetime(Data['Session end'])
    # filtering
    Data = Data.dropna(subset=['Type', 'Interval kWh', 'Interval average demand kW'])

    print("Number of Intervals in 2024:", len(Data))
    print("Test year 2024:", year)
    print("Date range 2024:", Data['Interval start'].min(), "to", Data['Interval start'].max())
    print("Unique sites 2024:", len(Data['Site Location'].unique()))
    Data['Car_'] = Data['10-digit UID'].astype(str)

ValueError: time data "02-02-2023 08:30:00" doesn't match format "%m/%d/%Y %H:%M", at position 616887. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.